In [0]:
data = [
    (101, "2024-06-01 10:00:00"),
    (101, "2024-06-01 10:10:00"),
    (101, "2024-06-01 11:00:00"),
    (102, "2024-06-01 09:00:00"),
    (102, "2024-06-01 09:20:00"),
    (102, "2024-06-01 10:00:00")
]

# Create DataFrame
df = spark.createDataFrame(data, ["user_id", "timestamp"])
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

salary_spec = Window.partitionBy('user_id').orderBy('timestamp')
df_res = df.withColumn('lag_date',lag('timestamp',1).over(salary_spec))\
    .withColumn('date_diff',(unix_timestamp(col('timestamp'))-unix_timestamp(col('lag_date')))/60)
display(df_res)

In [0]:
df_id = df_res.withColumn('new_id',lit(1))
df_id.show()

In [0]:
df_final = df_id.na.fill({"date_diff":0}).withColumn('final_id',when(col("date_diff")<30,col("new_id")).otherwise(col("new_id")+1))
display(df_final)

#second sample of data


In [0]:
import random
from datetime import datetime, timedelta

user_ids = [random.randint(100, 110) for _ in range(1000)]
base_date = datetime(2024, 6, 1, 8, 0, 0)
data = []
for uid in user_ids:
    # Each user gets between 1 and 5 timestamps
    n_times = random.randint(1, 5)
    last_time = base_date + timedelta(minutes=random.randint(0, 60*12))
    for _ in range(n_times):
        data.append((uid, last_time.strftime("%Y-%m-%d %H:%M:%S")))
        last_time += timedelta(minutes=random.randint(5, 120))

df = spark.createDataFrame(data, ["user_id", "timestamp"])
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = df.withColumn('timestamp', col("timestamp").cast("timestamp"))

window_spec = Window.partitionBy("user_id").orderBy("timestamp")

df = df.withColumn('prev_ts', lag("timestamp").over(window_spec))
df = df.withColumn('time_diff', (unix_timestamp("timestamp") - unix_timestamp("prev_ts")) / 60)

df = df.withColumn('new_session_flag', (col("time_diff") > 30).cast('integer'))
df = df.na.fill({'new_session_flag': 1})
df = df.withColumn('session_id',sum('new_session_flag').over(window_spec))

display(df)
